## Create sample data of just the body of articles for testing

In [1]:
import pandas as pd
import re
from bs4 import BeautifulSoup
from tqdm import tqdm
tqdm.pandas()

sample_data_path = "./sample_data/Articles_Nov_2020_March_2023.csv" # Using this as I don't have the other one

# Temporary. Use given article data set. Comment out when obtain the other data sate
full_df = pd.read_csv(sample_data_path)

# Format data set to match expected pipeline input
full_df = full_df.rename(columns={"Headline": "hl1", "Body": "body"})

# Make 'tagging' column be the id column
tagging_col = full_df.pop('Tagging')
full_df.insert(0, '_id', tagging_col)

# Drop rows where at least one of the specified columns is empty
columns_to_check = ['_id', 'hl1', 'body'] 
full_df = full_df.dropna(subset=columns_to_check, how='all')

# Drop empty rows too
full_df = full_df[~full_df['body'].apply(lambda x: isinstance(x, float))]
full_df = full_df[~full_df['hl1'].apply(lambda x: isinstance(x, float))]


### Pick out a sample of random articles

In [2]:
sample_count = 4

# Pick a random sample of articles
raw_df = full_df.sample(sample_count)
# raw_df = full_df
len(raw_df)


4

### Can also pick out known articles 

In [3]:
# List of known _id values (these I know have facility mentions)
known_ids = [
    '00000179-d37f-d770-a57f-f77f3c9c0001',
    '0000017a-8257-dc5c-a77a-8ad76dfe0001',
    '0000017d-1962-d269-a3fd-bf7b76d40001',
    '00000183-f771-d28d-a9f3-f7f9d78f0001'
]

# Comment out if not wanted
# raw_df = full_df[full_df['_id'].isin(known_ids)]
len(raw_df)

4

### Format Dataframe

In [4]:
df = pd.concat([raw_df['_id'], raw_df['hl1'], raw_df['body']], axis=1)

df = df.drop_duplicates(subset=['hl1'])

# Function to extract the text from the html of the article
func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
df['body'] = df['body'].progress_apply(func_clean_html)
df['hl1'] = df['hl1'].progress_apply(func_clean_html)

# Function to remove extra symbols from the text
func_clean_regex = lambda text: ' '.join([word for word in re.findall(r'[A-Za-z0-9!@#$%^&*().]+', text) if len(word) > 1])
df['body'] = df['body'].progress_apply(func_clean_regex)
df['hl1'] = df['hl1'].progress_apply(func_clean_regex)

100%|██████████| 4/4 [00:00<00:00, 1241.10it/s]


### Save Dataframe

In [5]:
df.to_csv('./sample_data/cleaned_sample_data.csv', index=False)